In [0]:
%sql
create or replace temp view vw_gold_claims_by_policy_type_and_status 
as
select
   policy_type,
   claim_status,
   count(*) as total_claim,
   sum(claim_amount) as total_claim_amount
FROM
   silverlayer.claim c
   Join silverlayer.policydata p
   on c.policy_id = p.policy_id
Group by
   policy_type,
   claim_status
having
  policy_type is not null;



In [0]:
%sql
select * from vw_gold_claims_by_policy_type_and_status

policy_type,claim_status,total_claim,total_claim_amount
Auto,Pending,11,1329891
Health,Denied,19,1787194
Auto,Completed,170,17148620
Health,In Progress,167,15659787
Health,Completed,149,14746926
Health,Pending,15,1801335
Auto,In Progress,159,15365911
Auto,Denied,11,1197860


In [0]:
%sql
MERGE INTO goldlayer.claims_by_policy_type_and_status as T USING vw_gold_claims_by_policy_type_and_status as s on 
t.policy_type = s.policy_type and t.claim_status = s.claim_status WHEN MATCHED THEN UPDATE SET t.total_claim = s.total_claim, t.total_claim_amount = s.total_claim_amount, t.update_timestamp = current_timestamp()
WHEN NOT MATCHED THEN INSERT (policy_type, claim_status, total_claim, total_claim_amount, update_timestamp)
values(s.policy_type, s.claim_status, s.total_claim, s.total_claim_amount, current_timestamp())

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
8,0,0,8


In [0]:
%sql
select * from goldlayer.claims_by_policy_type_and_status

policy_type,claim_status,total_claim,total_claim_amount,update_timestamp
Auto,Pending,11.0,1329891.0,2026-10-25T16:09:31.307331Z
Health,Denied,19.0,1787194.0,2026-10-25T16:09:31.307331Z
Auto,Completed,170.0,1.714862E7,2026-10-25T16:09:31.307331Z
Health,In Progress,167.0,1.5659787E7,2026-10-25T16:09:31.307331Z
Health,Completed,149.0,1.4746926E7,2026-10-25T16:09:31.307331Z
Health,Pending,15.0,1801335.0,2026-10-25T16:09:31.307331Z
Auto,In Progress,159.0,1.5365911E7,2026-10-25T16:09:31.307331Z
Auto,Denied,11.0,1197860.0,2026-10-25T16:09:31.307331Z
